In [ ]:
# ─────────────────────────────────────────────
# SAVE MODEL WEIGHTS FOR SUBMISSION
# ─────────────────────────────────────────────
import os
import pickle
import zipfile

# 1. Save fine-tuned MARBERT weights
MARBERT_SAVE_DIR = "./saved_marbert"
model.save_pretrained(MARBERT_SAVE_DIR)
tokenizer.save_pretrained(MARBERT_SAVE_DIR)
print(f"MARBERT saved to {MARBERT_SAVE_DIR}")

# 2. Save aspect classifier (TF-IDF + sklearn model + label binarizer)
ASPECT_CLF_PATH = "./aspect_classifier.pkl"
with open(ASPECT_CLF_PATH, "wb") as f:
    pickle.dump({
        "tfidf":      tfidf_asp,
        "clf":        aspect_clf,
        "mlb":        mlb_asp,
        "thresholds": ASPECT_THRESHOLDS,
    }, f)
print(f"Aspect classifier saved to {ASPECT_CLF_PATH}")

# 3. Save label encoder
LE_PATH = "./label_encoder.pkl"
with open(LE_PATH, "wb") as f:
    pickle.dump(le, f)
print(f"Label encoder saved to {LE_PATH}")

# 4. ZIP everything together
ZIP_PATH = "submission_with_weights.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # MARBERT weights folder
    for root, dirs, files in os.walk(MARBERT_SAVE_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, ".")
            zf.write(full_path, arcname)
    
    # Sklearn artifacts
    zf.write(ASPECT_CLF_PATH, "aspect_classifier.pkl")
    zf.write(LE_PATH, "label_encoder.pkl")
    
    # Your notebook
    zf.write("/kaggle/working/your_notebook.ipynb", "notebook.ipynb")  # adjust path
    
    # submission output
    zf.write(OUTPUT_PATH, "submission.json")

print(f"\n✅ ZIP created: {ZIP_PATH}")
print("Contents:")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for name in zf.namelist():
        print(f"  {name}")